# 🦅 PREDATOR Analytics v63.0-ELITE: Colab GPU Node
**Mode**: GPU-Accelerated Analytics
**Architecture**: K3s + Helm + GPU Passthrough + zrok

In [ ]:
# 1. Системна підготовка та GPU драйвери
!curl -sfL https://get.k3s.io | INSTALL_K3S_SKIP_ENABLE=true sh -
!curl https://raw.githubusercontent.com/helm/helm/main/scripts/get-helm-3 | bash
!wget -q https://github.com/openziti/zrok/releases/download/v0.4.42/zrok_0.4.42_linux_amd64.tar.gz && tar -xzf zrok_*.tar.gz && chmod +x zrok

import os, subprocess, time, threading

print("🚀 Запуск K3s у Colab (GPU Mode)...")
DATA_DIR = "/content/k3s-data"
KUBECONFIG = f"{DATA_DIR}/kubeconfig.yaml"
os.environ['KUBECONFIG'] = KUBECONFIG
os.makedirs(DATA_DIR, exist_ok=True)

k3s_cmd = [
    "k3s", "server",
    "--write-kubeconfig-mode", "644",
    "--data-dir", DATA_DIR,
    "--snapshotter", "native",
    "--disable", "traefik",
    "--disable", "servicelb"
]

subprocess.Popen(k3s_cmd, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)

for i in range(60):
    if os.path.exists(KUBECONFIG):
        check = subprocess.run(["/usr/local/bin/kubectl", "get", "nodes"], capture_output=True)
        if check.returncode == 0: break
    time.sleep(3)

# 2. Деплой
!git clone https://github.com/dima1203oleg/predator-analytics.git repo
os.chdir("/content/repo/deploy/helm/predator")

!/usr/local/bin/helm repo add bitnami https://charts.bitnami.com/bitnami
!/usr/local/bin/helm repo add redpanda https://charts.redpanda.com
!/usr/local/bin/helm repo update
!/usr/local/bin/helm dependency build

# GPU-specific values
gpu_values = """
global:
  domain: colab-gpu.predator.ua
coreApi:
  replicaCount: 1
  env: { VRAM_LIMIT_MB: "15000", CUDA_VISIBLE_DEVICES: "0" }
postgresql: { enabled: true }
neo4j: { enabled: true }
qdrant: { enabled: true }
"""
with open('values-gpu.yaml', 'w') as f: f.write(gpu_values)

!/usr/local/bin/helm install predator . -f values.yaml -f values-gpu.yaml --namespace predator-elite --create-namespace

# 3. zrok Тунель
!/content/zrok enable 1eeje4um7yvA

def start_tunnel():
    def _run():
        while True:
            check = subprocess.run(["/usr/local/bin/kubectl", "get", "svc", "-n", "predator-elite", "predator-core-api"], capture_output=True)
            if check.returncode == 0: break
            time.sleep(10)
        subprocess.Popen(["/usr/local/bin/kubectl", "port-forward", "-n", "predator-elite", "svc/predator-core-api", "8000:8000"])
        time.sleep(10)
        p = subprocess.Popen(["/content/zrok", "share", "public", "http://localhost:8000", "--headless"], stdout=subprocess.PIPE, text=True)
        for line in p.stdout:
            if "access your share at" in line:
                print(f"\n🔥 PREDATOR GPU NODE IS LIVE!\n🔗 URL: {line.split('access your share at')[-1].strip()}")
                break
    threading.Thread(target=_run, daemon=True).start()

start_tunnel()
!/usr/local/bin/kubectl get pods -n predator-elite